# LangChain Document Loaders and Text Splitters: PDF, Web, and Recursive Character Splitter

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/langchain_4)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q langchain langchain-openai langchain-community chromadb

In [ ]:
from langchain_core.documents import Document

doc = Document(
    page_content="LangChain is a framework for building LLM-powered applications.",
    metadata={"source": "intro.md", "page": 1, "author": "Soham Sharma"}
)

print(f"Content: {doc.page_content}")
print(f"Metadata: {doc.metadata}")
print(f"Type: {type(doc)}")

In [ ]:
from langchain_community.document_loaders import TextLoader
import tempfile
import os

# Create a temp file with sample content
sample_text = """Introduction to Transformers

The transformer architecture was introduced in 'Attention Is All You Need' (2017).
It relies entirely on attention mechanisms, discarding recurrence and convolutions.

Key components:
- Multi-head self-attention
- Feed-forward networks
- Positional encoding
- Layer normalization

The encoder-decoder structure made it ideal for translation tasks initially.
"""

with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write(sample_text)
    tmp_path = f.name

loader = TextLoader(tmp_path, encoding='utf-8')
docs = loader.load()

print(f"Number of documents: {len(docs)}")
print(f"Content length: {len(docs[0].page_content)} chars")
print(f"Metadata: {docs[0].metadata}")
print(f"\nFirst 100 chars: {docs[0].page_content[:100]}")

os.unlink(tmp_path)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# PyPDFLoader: one Document per page
# Install: pip install pypdf
loader = PyPDFLoader("path/to/document.pdf")
pages = loader.load()

print(f"Pages loaded: {len(pages)}")
print(f"Page 0 metadata: {pages[0].metadata}")
print(f"Page 0 content (first 200 chars): {pages[0].page_content[:200]}")

In [ ]:
from langchain_community.document_loaders import PDFMinerLoader

# PDFMinerLoader: better layout preservation, fewer encoding issues
# Install: pip install pdfminer.six
loader = PDFMinerLoader("path/to/complex_layout.pdf")
docs = loader.load()  # returns single Document with full text

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
import bs4

# WebBaseLoader: loads and parses HTML pages
# Install: pip install beautifulsoup4
loader = WebBaseLoader(
    web_paths=["https://python.langchain.com/docs/introduction/"],
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("article-content", "markdown", "content")
        )
    ),
)

# Note: requires internet access
# docs = loader.load()
# print(f"Loaded {len(docs)} document(s)")
# print(f"Content length: {len(docs[0].page_content)} chars")

print("WebBaseLoader configured — call loader.load() to fetch content.")
print("bs_kwargs filters HTML to only the content sections (reduces noise).")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

text = """
Transformers and Attention Mechanisms

The transformer architecture revolutionized natural language processing. 
At its core is the self-attention mechanism, which allows each token to 
attend to every other token in the sequence.

Multi-Head Attention

Multi-head attention runs several attention operations in parallel. Each 
"head" learns to attend to different aspects of the input — one head might 
focus on syntactic relationships while another captures semantic similarity.

The output of all heads is concatenated and projected through a linear layer.
This allows the model to jointly attend to information from different 
representation subspaces at different positions.

Feed-Forward Networks

After attention, each position passes through the same feed-forward network 
independently. This consists of two linear transformations with a ReLU 
activation: FFN(x) = max(0, xW1 + b1)W2 + b2.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
)

chunks = splitter.split_text(text)
print(f"Text length: {len(text)} chars")
print(f"Number of chunks: {len(chunks)}")
print(f"\n--- Chunk 0 ({len(chunks[0])} chars) ---")
print(chunks[0])
print(f"\n--- Chunk 1 ({len(chunks[1])} chars) ---")
print(chunks[1])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

docs = [
    Document(
        page_content="""Introduction to Neural Networks

Neural networks are computational models inspired by biological brains. 
They consist of layers of interconnected nodes that process information.
Each connection has a weight that is adjusted during training.""",
        metadata={"source": "chapter1.pdf", "page": 1}
    ),
    Document(
        page_content="""Backpropagation Algorithm

The backpropagation algorithm computes gradients of the loss function 
with respect to the weights using the chain rule of calculus.
These gradients are used to update weights via gradient descent.""",
        metadata={"source": "chapter2.pdf", "page": 5}
    ),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
chunks = splitter.split_documents(docs)

print(f"Input: {len(docs)} documents → Output: {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i}: {len(chunk.page_content)} chars | source: {chunk.metadata['source']}, page: {chunk.metadata['page']}")
    print(f"  {chunk.page_content[:80]}...")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def estimate_chunk_token_count(text: str, chars_per_token: float = 4.0) -> int:
    """Rough estimate: ~4 chars per token for English text."""
    return int(len(text) / chars_per_token)

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

sample = "The attention mechanism was introduced by Bahdanau et al. in 2014 for neural machine translation. " * 20
chunks = splitter.split_text(sample)

for i, chunk in enumerate(chunks[:3]):
    est_tokens = estimate_chunk_token_count(chunk)
    print(f"Chunk {i}: {len(chunk)} chars ≈ {est_tokens} tokens")

In [ ]:
from langchain_text_splitters import SentenceTransformersTokenTextSplitter

splitter = SentenceTransformersTokenTextSplitter(
    model_name="sentence-transformers/all-mpnet-base-v2",
    chunk_overlap=0,
    tokens_per_chunk=256,  # max tokens per chunk (model-specific)
)

text = "The transformer architecture uses self-attention. It was introduced in 2017. The paper proposed both encoder and decoder stacks. Each layer has multi-head attention followed by feed-forward networks."
chunks = splitter.split_text(text)
for i, c in enumerate(chunks):
    print(f"Chunk {i}: {c}")

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List
import tempfile
import os

def ingest_documents(file_paths: List[str], chunk_size: int = 800, chunk_overlap: int = 100) -> List[Document]:
    """
    Load and split documents from file paths.
    Returns a list of chunked Documents with source metadata.
    """
    raw_docs = []

    for path in file_paths:
        ext = os.path.splitext(path)[1].lower()

        if ext in ['.txt', '.md']:
            loader = TextLoader(path, encoding='utf-8')
            raw_docs.extend(loader.load())
        else:
            raise ValueError(f"Unsupported file type: {ext}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = splitter.split_documents(raw_docs)
    print(f"Loaded {len(raw_docs)} docs → split into {len(chunks)} chunks")
    return chunks

# Test with sample files
content_a = "Attention mechanisms allow models to focus on relevant parts of the input.\n\n" + "They compute a weighted sum of values based on query-key similarity.\n\n" * 5
content_b = "Transformers use positional encoding to inject sequence order information.\n\n" + "This allows parallel processing unlike recurrent networks.\n\n" * 5

with tempfile.TemporaryDirectory() as tmpdir:
    path_a = os.path.join(tmpdir, "attention.txt")
    path_b = os.path.join(tmpdir, "transformers.txt")
    open(path_a, 'w').write(content_a)
    open(path_b, 'w').write(content_b)

    chunks = ingest_documents([path_a, path_b], chunk_size=200, chunk_overlap=30)
    for chunk in chunks[:3]:
        print(f"\n[{chunk.metadata['source'].split('/')[-1]}] {chunk.page_content[:80]}...")